In [1]:
import os

In [2]:
%pwd

'c:\\Users\\chauh\\OneDrive\\Desktop\\Text-Summarization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\chauh\\OneDrive\\Desktop\\Text-Summarization'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [6]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE__PATH):
        
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)


        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config=self.config.model_trainer
        params=self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config=ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            evaluation_strategy=params.evaluation_strategy,
            eval_steps=params.evaluation_strategy,
            save_steps=params.save_steps,
            gradient_accumulation_steps=params.gradient_accumulation_steps


        )

        return model_trainer_config
    

In [7]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset,load_from_disk
import torch

[2026-04-30 20:38:38,751: WARNING:  module_wrapper: From c:\Users\chauh\miniconda3\envs\codexenv\Lib\site-packages\tf_keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.
 ]
[2026-04-30 20:38:39,706: INFO:  config: TensorFlow version 2.21.0 available. ]


In [8]:
class ModelTrainer:
    def __init__(self,config: ModelTrainerConfig):
        self.config=config

    
    def train(self):
        device="cuda" if torch.cuda.is_available() else "cpu"
        tokenizer=AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator=DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)

        #loadding data
        dataset_samsum_pt=load_from_disk(self.config.data_path)

    
        # trainer_args=TrainingArguments(
        #     output_dir=self.config.root_dir,num_train_epochs=self.config.num_train_epochs, warmup_steps=self.config.warmup_steps,
        #     per_device_train_batch_size=self.config.per_device_train_batch_size,per_device_eval_batch_size=self.config.per_device_train_batch_size,
        #     weight_decay=self.config.weight_decay,logging_steps=self.config.logging_steps,
        #     evaluation_strategy=self.config.evaluation_strategy,eval_steps=self.config.eval_steps,
        #     gradient_accumulation_steps=self.config.gradient_accumulation_steps
        # )



        training_args = TrainingArguments(
        output_dir=self.config.root_dir,
        num_train_epochs=1,
        warmup_steps=5,              # Very low
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        weight_decay=0.01,
        logging_steps=1,             # FORCE it to log every single step
        eval_strategy='no',          # Disable evaluation to save RAM
        save_steps=500,
        gradient_accumulation_steps=1, # Disable accumulation for now
        no_cuda=True,                # Force CPU
        max_steps=10                 # Just run 10 steps to see if it works!
        )

        # training_args=TrainingArguments(
        #     output_dir=self.config.root_dir,num_train_epochs=1,warmup_steps=500,
        #     per_device_train_batch_size=1,per_device_eval_batch_size=1,
        #     weight_decay=0.01, logging_steps=10,
        #     eval_strategy='steps',eval_steps=500, save_steps=1e6,
        #     gradient_accumulation_steps=16,dataloader_pin_memory=False

    #     training_args = TrainingArguments(
    #     output_dir=self.config.root_dir,
    #     num_train_epochs=1,
    #     warmup_steps=50,             # Reduced from 500
    #     per_device_train_batch_size=1,
    #     per_device_eval_batch_size=1,
    #     weight_decay=0.01,
    #     logging_steps=10,
    #     eval_strategy='steps',
    #     eval_steps=500,
    #     save_steps=1e6,
    #     gradient_accumulation_steps=16,
    #     fp16=False,                 # Ensure this is False for CPU
    #     dataloader_pin_memory=False, # STOP trying to use accelerator memory
    #     no_cuda=True                # Force CPU mode to avoid searching for GPU
    # )
    #     training_args = TrainingArguments(
    #  output_dir=self.config.root_dir,
    # # ...
    #     )

        trainer = Trainer(
            model=model_pegasus,
            args=training_args,  # <--- FIXED: Now matches the variable name above
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        # trainer=Trainer(model=model_pegasus,args=trainer_args,
        #                 tokeinzer=tokenizer,data_collator=seq2seq_data_collator,
        #                 train_dataset=dataset_samsum_pt["train"],
        #                 eval_dataset=dataset_samsum_pt["validation"])
        
        trainer.train()

        # savemodel
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))
        ## Save tokenizer 
        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))


                                            

In [9]:
try:
    config= ConfigurationManager()
    model_trainer_config=config.get_model_trainer_config()
    model_trainer_config=ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()

except Exception as e:
    raise e

[2026-04-30 20:38:45,726: INFO:  common: yaml file: config\config.yaml loaded successfully ]
[2026-04-30 20:38:45,732: INFO:  common: yaml file: params.yaml loaded successfully ]
[2026-04-30 20:38:45,734: INFO:  common: created directory at: artifacts ]
[2026-04-30 20:38:45,736: INFO:  common: created directory at: artifacts/model_trainer ]


config.json: 0.00B [00:00, ?B/s]

c:\Users\chauh\miniconda3\envs\codexenv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chauh\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

c:\Users\chauh\miniconda3\envs\codexenv\Lib\site-packages\transformers\training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
C:\Users\chauh\AppData\Local\Temp\ipykernel_26496\4282919744.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
1,4.348000
2,3.092200
3,3.489900
4,3.112100
5,2.495200
6,1.960500
7,3.513000
8,4.307600
9,3.118100
10,4.593400
